# GGA Module — Incident Typologies from Crash Narratives

Discover *nameable incident types* in BikeMaps crash narratives using text
embeddings (ArcGIS Pro 3.7 GeoAI toolbox), then validate whether those types
carry different injury risk. Everything runs in this one notebook inside
ArcGIS Pro — no R, no GPU.

**Requirements:** ArcGIS Pro 3.7, Advanced license, Deep Learning Libraries,
signed in to ArcGIS Online (one model download from Living Atlas).

## How to run this notebook (read first — Pro notebooks can jam)

- **Run cells one at a time, top to bottom. Never "Run All".**
- Every heavy cell prints a start timestamp and an expected duration, and
  **saves its result to disk** when it finishes. A finished step is recorded
  in `progress.json`.
- Re-running a finished cell **skips itself** — outputs on disk are reused.
- **If a cell hangs** far past its expected duration: geoprocessing tools often
  cannot be interrupted from the notebook. Save, close ArcGIS Pro (Task
  Manager → end ArcGISPro.exe if needed), reopen, re-run **cell 0 only**,
  check the printed checklist, and continue from the first unchecked step.
  You lose at most the one step that was running.
- Keep intermediate layers **out of the map** until the final step — an open
  layer holds a schema lock that makes re-runs hang.


## 0 — Setup, config, and the progress tracker

Edit the four paths, then run. Safe to re-run any time; it also prints the
checklist of completed steps.

In [ ]:
# ========================== EDIT THESE FOUR PATHS ==========================
HOME      = r"E:\GGA\Embeddings\typology_module"              # working folder (created if missing)
CANADA    = r"E:\GGA\data\20240425Canada.xlsx"                # BikeMaps Canada export
US        = r"E:\GGA\data\20240425United States.xlsx"         # BikeMaps US export
TEXT_DLPK = r"E:\GGA\Embeddings\TextEncoder.dlpk"             # from the smoke test; auto-downloaded in step 3 if missing
# ===========================================================================

import os, re, json, time, random
import arcpy
import numpy as np
import pandas as pd

SEED = 1208
random.seed(SEED); np.random.seed(SEED)

os.makedirs(HOME, exist_ok=True)
GDB      = os.path.join(HOME, "typology.gdb")
PROGRESS = os.path.join(HOME, "progress.json")
STEPS    = ["1_prep", "2_points", "3_model", "4_embed", "5_extract",
            "6_cluster", "7_name", "8_validate", "9_map"]

def _load():
    try:
        with open(PROGRESS) as f:
            return json.load(f)
    except Exception:
        return {}

def done(step):
    return _load().get(step, {}).get("done", False)

def mark(step, **info):
    p = _load()
    p[step] = {"done": True, "at": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
    with open(PROGRESS, "w") as f:
        json.dump(p, f, indent=1)
    print(f"[done] {step}", flush=True)

def begin(step, expected):
    print(f"[start] {step} at {time.strftime('%H:%M:%S')} — expected {expected}", flush=True)
    return time.time()

def status():
    p = _load()
    print("Progress:", flush=True)
    for s in STEPS:
        box = "[x]" if p.get(s, {}).get("done") else "[ ]"
        print(f"  {box} {s}", flush=True)

if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(HOME, "typology.gdb")
arcpy.env.workspace = GDB
arcpy.env.overwriteOutput = True

import sklearn
print("ArcGIS Pro", arcpy.GetInstallInfo()["Version"], "| license:", arcpy.ProductInfo(), flush=True)
print("sklearn", sklearn.__version__, "| pandas", pd.__version__, flush=True)
HAS_HDBSCAN = tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 3)
print("HDBSCAN available in sklearn:", HAS_HDBSCAN, "(KMeans fallback otherwise)", flush=True)
status()

## 1 — Load the reports, build the outcome, mask the leakage

We keep personally-involved incidents after 2016-11-30 with a narrative of at
least 20 characters, and derive `injured` (any injury vs. none).

**Why masking?** 30% of injured riders' narratives contain outcome words
(hospital, fracture, pain...) vs 6% of uninjured — P(injured | outcome word)
is 51% against a 17% base rate. Clustering raw text would discover an "injury
stories" cluster, making the injury-rate validation circular. Masking deletes
every *sentence* containing outcome vocabulary, so clusters can only form
around circumstances. ~30 s.

In [ ]:
STEP = "1_prep"
PREP_CSV = os.path.join(HOME, "narratives.csv")

if done(STEP) and os.path.exists(PREP_CSV):
    narr = pd.read_csv(PREP_CSV)
    print(f"skipped — already done ({len(narr)} rows)", flush=True)
else:
    t0 = begin(STEP, "~30 s")
    frames = []
    for country, path in [("CA", CANADA), ("US", US)]:
        t = pd.read_excel(path)
        t["country"] = country
        frames.append(t)
        print(f"  read {country}: {len(t)} reports", flush=True)
    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    d = df[(df["date"] > "2016-11-30")
           & (df["personal_involvement"] == "Yes")
           & df["ebike"].isin(["Yes", "No"])].copy()
    d = d[d["details"].notna()
          & (d["details"].astype(str).str.len() >= 20)
          & d["x"].notna() & d["y"].notna()]
    d["injured"] = ((~d["injury"].isin(["No injury", "Unknown"]))
                    & d["injury"].notna()).astype(int)

    # Outcome lexicon — frozen before clustering. Extend only by reading
    # narratives, never by peeking at cluster results.
    LEAK = re.compile(
        r"\b(injur\w*|hospital\w*|doctor|ambulance|paramedic|brok\w*|fractur\w*"
        r"|pain\w*|bruis\w*|stitch\w*|concuss\w*|emergency|x-?ray|sprain\w*"
        r"|bleed\w*|blood|scrape\w*|road ?rash|wound\w*|surgery|clinic|e\.?r\.?)\b",
        re.I)

    def mask_text(s):
        sentences = re.split(r"(?<=[.!?])\s+", str(s))
        kept = [x for x in sentences if not LEAK.search(x)]
        return " ".join(kept).strip() or "no description"

    d["details_masked"] = d["details"].astype(str).apply(mask_text)
    d["was_masked_empty"] = (d["details_masked"] == "no description").astype(int)

    narr = d[["pk", "country", "x", "y", "ebike", "injured",
              "details", "details_masked", "was_masked_empty"]].reset_index(drop=True)
    narr.to_csv(PREP_CSV, index=False, encoding="utf-8-sig")
    print(f"{len(narr)} narratives | injured {narr.injured.mean():.1%} "
          f"| fully masked {narr.was_masked_empty.mean():.1%} "
          f"| {time.time()-t0:.0f}s", flush=True)
    mark(STEP, rows=len(narr))

## 2 — Narratives to map points (~10 s)

In [ ]:
STEP = "2_points"
FC = os.path.join(GDB, "narratives")

if done(STEP) and arcpy.Exists(FC):
    print("skipped — already done", flush=True)
else:
    t0 = begin(STEP, "~10 s")
    arcpy.management.XYTableToPoint(PREP_CSV, FC, "x", "y",
                                    coordinate_system=arcpy.SpatialReference(4326))
    n = int(arcpy.management.GetCount(FC)[0])
    assert n == len(narr), f"feature count {n} != {len(narr)} rows"
    print(f"{n} point features | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, n=n)

## 3 — Text embedding model from Living Atlas

Downloads the *HF Text Feature Extraction* package (sentence-transformer,
384-dim) if the configured path doesn't already hold it. Needs your ArcGIS
Online sign-in. First time: a few minutes, depending on connection.

In [ ]:
STEP = "3_model"

if os.path.exists(TEXT_DLPK):
    print("model already on disk:", TEXT_DLPK, flush=True)
    if not done(STEP):
        mark(STEP, path=TEXT_DLPK)
else:
    t0 = begin(STEP, "2-10 min (download)")
    from arcgis.gis import GIS
    gis = GIS("home")
    print("  signed in as", gis.users.me.username, flush=True)
    hits = gis.content.search('title:"HF Text Feature Extraction" owner:esri_analytics',
                              item_type="Deep Learning Package") or \
           gis.content.search('title:"HF Text Feature Extraction"',
                              item_type="Deep Learning Package", outside_org=True)
    assert hits, "Living Atlas item not found — check your sign-in/subscription"
    TEXT_DLPK = hits[0].download(save_path=os.path.dirname(TEXT_DLPK) or HOME)
    print("  downloaded ->", TEXT_DLPK, f"| {time.time()-t0:.0f}s", flush=True)
    mark(STEP, path=TEXT_DLPK)

## 4 — Generate embeddings for the **masked** narratives

The long cell. CPU-only is fine; budget roughly **10-40 min** for a few
thousand narratives (your smoke test embedded 60 in ~2 min on CPU — scale
accordingly). The start timestamp prints immediately; the geoprocessing tool
prints nothing until it finishes. If Pro dies mid-run, delete the partial
`narratives_emb` feature class after restarting, then re-run this cell.

In [ ]:
STEP = "4_embed"
EMB_FC = os.path.join(GDB, "narratives_emb")

if done(STEP) and arcpy.Exists(EMB_FC):
    print("skipped — already done", flush=True)
else:
    if arcpy.Exists(EMB_FC):
        arcpy.management.Delete(EMB_FC)   # clear a partial output from a crashed run
        print("  deleted partial output from an earlier run", flush=True)
    t0 = begin(STEP, "10-40 min on CPU — go get coffee")
    arcpy.geoai.GenerateEmbeddingsUsingAIModels(
        FC, EMB_FC, TEXT_DLPK,
        [["text_field", "details_masked"], ["batch_size", "16"]])
    fields = {f.name: f.type for f in arcpy.ListFields(EMB_FC)}
    assert fields.get("Embedding") == "Blob", "no Blob field named Embedding"
    print(f"embedded {arcpy.management.GetCount(EMB_FC)[0]} narratives "
          f"| {(time.time()-t0)/60:.1f} min", flush=True)
    mark(STEP)

## 5 — Unpack embeddings to a numeric matrix (~1-5 min)

Extract Embedding To Fields turns the blob into `emb_*` columns (the slow part
is adding ~384 fields to the geodatabase); we then pull them into numpy and
save `X.npy` so clustering never needs ArcGIS again.

In [ ]:
STEP = "5_extract"
FLAT_FC = os.path.join(GDB, "narratives_emb_flat")
X_NPY   = os.path.join(HOME, "X.npy")
PK_CSV  = os.path.join(HOME, "pk_order.csv")

if done(STEP) and os.path.exists(X_NPY):
    X = np.load(X_NPY)
    pk_order = pd.read_csv(PK_CSV)["pk"].to_numpy()
    print(f"skipped — already done (X {X.shape})", flush=True)
else:
    t0 = begin(STEP, "1-5 min")
    if arcpy.Exists(FLAT_FC):
        arcpy.management.Delete(FLAT_FC)
    arcpy.geoai.ExtractEmbeddingToFields(EMB_FC, FLAT_FC, "Embedding", "emb")
    emb_cols = [f.name for f in arcpy.ListFields(FLAT_FC)
                if f.name.lower().startswith("emb") and f.type in ("Double", "Single")]
    emb_cols.sort(key=lambda c: int(re.sub(r"\D", "", c) or 0))
    rows, pks = [], []
    with arcpy.da.SearchCursor(FLAT_FC, ["pk"] + emb_cols) as cur:
        for r in cur:
            pks.append(r[0]); rows.append(r[1:])
    X = np.asarray(rows, dtype=np.float32)
    pk_order = np.asarray(pks)
    np.save(X_NPY, X)
    pd.DataFrame({"pk": pk_order}).to_csv(PK_CSV, index=False)
    print(f"X {X.shape} ({len(emb_cols)} dims) saved | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, dims=X.shape[1])

# align the narrative table to the embedding row order
narr_x = narr.set_index("pk").loc[pk_order].reset_index()
assert len(narr_x) == X.shape[0]

## 6 — Cluster: PCA then HDBSCAN (KMeans fallback) (~1 min)

L2-normalize so distance reflects meaning, PCA to 15 components, then HDBSCAN
with `min_cluster_size=30`. HDBSCAN chooses the number of clusters itself and
leaves genuinely odd narratives unclustered (label −1) instead of forcing
them — that noise group is a feature, not a bug. A quick bootstrap stability
check follows: clusters that dissolve under resampling don't get reported.

In [ ]:
STEP = "6_cluster"
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

t0 = begin(STEP, "~1 min")
Xn = normalize(X)                       # L2 per row
Z = PCA(n_components=15, random_state=SEED).fit_transform(Xn)

if HAS_HDBSCAN:
    from sklearn.cluster import HDBSCAN
    model = HDBSCAN(min_cluster_size=30)
    labels = model.fit_predict(Z)
    method = "HDBSCAN(min_cluster_size=30)"
else:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    best = None
    for k in range(5, 16):
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z)
        s = silhouette_score(Z, km.labels_)
        print(f"  k={k:2d} silhouette={s:.3f}", flush=True)
        if best is None or s > best[0]:
            best = (s, k, km.labels_)
    labels = best[2]; method = f"KMeans(k={best[1]})"

narr_x["cluster"] = labels
counts = narr_x["cluster"].value_counts().sort_index()
print(f"\n{method}", flush=True)
print(counts.to_string(), flush=True)
if -1 in counts.index:
    print(f"(-1 = unclustered noise: {counts[-1]} narratives)", flush=True)
narr_x.to_csv(os.path.join(HOME, "narratives_clustered.csv"), index=False, encoding="utf-8-sig")
print(f"{time.time()-t0:.0f}s", flush=True)
mark(STEP, method=method, clusters=int((counts.index >= 0).sum()))

In [ ]:
# 6b — stability check (optional but recommended, ~2 min)
from sklearn.metrics import adjusted_rand_score

if HAS_HDBSCAN:
    t0 = begin("6b_stability", "~2 min")
    rng = np.random.default_rng(SEED)
    aris = []
    for b in range(10):
        idx = rng.choice(len(Z), size=int(0.8 * len(Z)), replace=False)
        lb = HDBSCAN(min_cluster_size=30).fit_predict(Z[idx])
        aris.append(adjusted_rand_score(labels[idx], lb))
    print(f"bootstrap ARI: mean {np.mean(aris):.2f} (min {min(aris):.2f}, max {max(aris):.2f})", flush=True)
    print("rule of thumb: mean ARI > 0.5 = stable enough to name; below that, "
          "raise min_cluster_size and re-run step 6", flush=True)
else:
    print("stability check written for HDBSCAN; for KMeans compare k across seeds", flush=True)

## 7 — Name the clusters (the human-in-the-loop step)

For each cluster: top TF-IDF terms versus the rest of the corpus, the 5
narratives nearest the cluster center, and 5 random members. **Read them and
write a one-line definition in the `CLUSTER_NAMES` dict of the next cell.**
A cluster you can't name coherently stays unnamed — that's an honest result.

In [ ]:
STEP = "7_name"
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = begin(STEP, "~30 s compute, then reading time")
vec = TfidfVectorizer(stop_words="english", max_features=5000,
                      ngram_range=(1, 2), min_df=5)
T = vec.fit_transform(narr_x["details_masked"])
terms = np.asarray(vec.get_feature_names_out())

for c in sorted(set(labels)):
    if c == -1:
        continue
    in_c = labels == c
    score = T[in_c].mean(axis=0).A1 - T[~in_c].mean(axis=0).A1
    top = terms[np.argsort(score)[::-1][:12]]
    center = Z[in_c].mean(axis=0)
    dist = np.linalg.norm(Z[in_c] - center, axis=1)
    nearest = narr_x.loc[np.where(in_c)[0][np.argsort(dist)[:5]], "details_masked"]
    print("=" * 70, flush=True)
    print(f"CLUSTER {c}  (n={in_c.sum()}, injured {narr_x.loc[in_c,'injured'].mean():.0%})", flush=True)
    print("terms:", ", ".join(top), flush=True)
    for s in nearest:
        print("  •", str(s)[:180].replace("\n", " "), flush=True)
    sample = narr_x.loc[in_c].sample(min(5, in_c.sum()), random_state=SEED)["details_masked"]
    for s in sample:
        print("  ○", str(s)[:180].replace("\n", " "), flush=True)
mark(STEP)

In [ ]:
# 7b — write your one-line names here after reading, then run
CLUSTER_NAMES = {
    # 0: "Right-hook / turning vehicle at intersection",
    # 1: "Dooring",
    # 2: "Surface hazard (gravel, ice, tracks)",
    # -1 stays "unclustered"
}
narr_x["typology"] = narr_x["cluster"].map(
    lambda c: CLUSTER_NAMES.get(c, "unclustered" if c == -1 else f"cluster {c}"))
print(narr_x["typology"].value_counts().to_string(), flush=True)

## 8 — Validate: does injury risk differ by typology? (~10 s)

Injury rate per typology with 95% Wilson intervals, e-bike share, and a
chi-square test against the pooled rate. Because clustering used **masked**
text, a significant difference here is about mechanism, not about riders
describing their injuries.

In [ ]:
STEP = "8_validate"
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint

t0 = begin(STEP, "~10 s")
rows = []
for c, g in narr_x[narr_x["cluster"] >= 0].groupby("typology"):
    lo, hi = proportion_confint(g["injured"].sum(), len(g), method="wilson")
    rows.append({"typology": c, "n": len(g),
                 "injury_rate": g["injured"].mean(),
                 "ci_low": lo, "ci_high": hi,
                 "ebike_share": (g["ebike"] == "Yes").mean()})
summary = pd.DataFrame(rows).sort_values("injury_rate", ascending=False)
print(summary.to_string(index=False,
      formatters={c: "{:.1%}".format for c in
                  ["injury_rate", "ci_low", "ci_high", "ebike_share"]}), flush=True)

ct = pd.crosstab(narr_x.loc[narr_x["cluster"] >= 0, "typology"],
                 narr_x.loc[narr_x["cluster"] >= 0, "injured"])
chi2, p, dof, _ = chi2_contingency(ct)
print(f"\nchi-square: chi2={chi2:.1f}, dof={dof}, p={p:.2g}", flush=True)
print(f"pooled injury rate: {narr_x['injured'].mean():.1%}", flush=True)
summary.to_csv(os.path.join(HOME, "typology_summary.csv"), index=False)
mark(STEP)

## 9 — Put the typologies on the map

Writes the cluster label and typology name back to the point feature class and
adds it to the open map. Symbolize by `typology` (unique values) — this is the
module's headline graphic. Do this **last**: once the layer is on the map it
holds a schema lock.

In [ ]:
STEP = "9_map"
t0 = begin(STEP, "~30 s")
lab = dict(zip(narr_x["pk"], zip(narr_x["cluster"], narr_x["typology"])))

for fld, typ in [("cluster_id", "LONG"), ("typology", "TEXT")]:
    if fld not in [f.name for f in arcpy.ListFields(FC)]:
        arcpy.management.AddField(FC, fld, typ, field_length=80)
with arcpy.da.UpdateCursor(FC, ["pk", "cluster_id", "typology"]) as cur:
    for r in cur:
        c, name = lab.get(r[0], (-1, "unclustered"))
        cur.updateRow([r[0], int(c), name])

try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    aprx.activeMap.addDataFromPath(FC)
    print("layer added to the active map — symbolize by 'typology'", flush=True)
except Exception as e:
    print("could not add to a map (running outside a Pro project?):", e, flush=True)
mark(STEP)
status()

## 10 — Optional exercise: see the leakage with your own eyes

Repeat steps 4-6 on the **raw** `details` field (change `text_field` to
`details` and the output names to `*_raw`). Compare the cluster tables: the
raw-text run grows a cluster whose top TF-IDF terms are *hospital, broken,
pain* and whose injury rate towers over the rest. That cluster is not an
incident type — it's the outcome leaking into the features. This is the most
convincing leakage demonstration a student will see.

## 11 — Optional: from typology to odds ratio

For 2-4 risk-distinct, nameable typologies, build per-incident binary flags
(keyword rules, validated on 100 hand-labeled narratives) and add them to a
logistic regression next to the survey fields (`statsmodels.api.Logit`). The
flags earn odds ratios and confidence intervals; the embeddings never appear
in the final model. That is the step that turns "the text contains signal"
into "dooring carries 2.3x the injury odds".